**Run with:** 

---

# MODIS Composite Comparison — Valencia 2024

**Question:** How do the 1‑day (F1C), 2‑day (F2), and 3‑day (F3) MODIS flood
composites differ over the same event?

Per the [MCDWD User Guide](https://www.earthdata.nasa.gov/s3fs-public/2025-12/MCDWD_VCDWD_UserGuide_RevF.pdf),
longer composites suppress cloud-shadow false-positives at the cost of temporal
sharpness. This notebook makes that trade-off visible with a few clean plots.

- **F1C** — 1‑day with cloud-shadow screening (sharpest, most noise)
- **F2** — 2‑day max-composite (recommended default)
- **F3** — 3‑day max-composite (smoothest, least noise)

In [1]:
from __future__ import annotations

import subprocess
import os
import sys
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rioxarray as rxr
import xarray as xr

# Resolve repo root from the notebook location
NOTEBOOK_DIR = Path.cwd()
candidate = NOTEBOOK_DIR
while candidate != candidate.parent and not (candidate / "pyproject.toml").exists():
    candidate = candidate.parent
REPO_ROOT = candidate
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 150, "font.size": 9})

print(f"REPO_ROOT = {REPO_ROOT}")
print(f"notebook  = {NOTEBOOK_DIR.relative_to(REPO_ROOT)}")

## 1. Fetch MODIS data for each composite

We use the same AOI and date window for all three composites so the only
variable is the compositing rule. The `--no-classify` flag preserves the raw
MCDWD pixel codes (0/1/2/3/255); `--harmonise` snaps everything to the
canonical 1‑arcmin grid so the rasters are pixel-aligned and directly
comparable.

**Prerequisite:** `EARTHDATA_TOKEN` must be set in `.env` at the repo root.

In [2]:
BBOX = "-1.5 38.8 0.5 40.0"
START = "2024-10-29"
END = "2024-11-04"
DATA_DIR = REPO_ROOT / "notebooks" / "drafts" / "data" / "modis_composites"
COMPOSITES = {
    "F1C": "1-day cloud-shadow screened",
    "F2": "2-day max-composite",
    "F3": "3-day max-composite",
}

print(f"Output root: {DATA_DIR.relative_to(REPO_ROOT)}")

In [3]:
for comp, desc in COMPOSITES.items():
    out_dir = DATA_DIR / comp
    harmonised_glob = list((out_dir / "modis" / "harmonised").glob("*_harmonised.tif"))
    if harmonised_glob:
        print(f"[{comp}] already fetched → {len(harmonised_glob)} file(s)")
        continue

    cmd = [
        "python", "-m", "atlantis.cli", "fetch",
        "--event", "Valencia_2024",
        "--source", "modis",
        "--bbox", BBOX,
        "--start-date", START,
        "--end-date", END,
        "--modis-backend", "laads_hdf4",
        "--modis-composite", comp,
        "--strategy", "peak",
        "--no-classify",
        "--harmonise",
        "--no-keep-processed",
        "--output", str(out_dir),
    ]
    print(f"[{comp}] fetching ({desc}) …")
    subprocess.run(cmd, cwd=REPO_ROOT, check=True, env={**os.environ, "PYTHONPATH": str(SRC)})

## 2. Load the harmonised rasters

Each composite writes a single `*_harmonised.tif` under its output tree — the
peak‑date flood layer resampled to 1‑arcmin via `mode` (majority class).
Because the harmoniser always snaps to the same canonical grid, the three
rasters are pixel‑aligned and we can compare them directly.

In [4]:
MOD_CODE = {
    0: "No water",
    1: "Surface water",
    2: "Recurring flood",
    3: "Unusual flood",
    255: "Insufficient data",
}

rasters: dict[str, xr.DataArray] = {}
peak_dates: dict[str, str] = {}

for comp in COMPOSITES:
    out_dir = DATA_DIR / comp
    tifs = sorted((out_dir / "modis" / "harmonised").glob("*_harmonised.tif"))
    if not tifs:
        raise FileNotFoundError(f"No harmonised output for {comp}")

    da = rxr.open_rasterio(tifs[0]).squeeze(drop=True)
    rasters[comp] = da

    # Filename: Valencia_2024_2024-11-03_modis_harmonised.tif
    date_part = tifs[0].stem.split("_")[1]  # YYYY-MM-DD
    peak_dates[comp] = date_part

    vals = da.values
    uniq, cnts = np.unique(vals, return_counts=True)
    total = vals.size
    print(f"\n[{comp}]  {tifs[0].name}")
    print(f"        shape={da.shape}  dtype={da.dtype}  peak_date={peak_dates[comp]}")
    for code, count in sorted(zip(uniq, cnts), key=lambda x: -x[1]):
        label = MOD_CODE.get(int(code), "?")
        print(f"        code {int(code):>3}  {count:>8,} px ({100*count/total:.1f}%)  {label}")


## 3. Pixel-value distributions

A quick histogram per composite, side by side. The key comparison:
- **Code 3 (unusual flood)** — how much flood signal does each composite capture?
- **Code 255 (insufficient data)** — how many pixels are masked out?

In [5]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)
COLORS = {0: "#d4c5a9", 1: "#2980b9", 2: "#f39c12", 3: "#e74c3c", 255: "#95a5a6"}
ORDER = [0, 1, 2, 3, 255]

for ax, (comp, da) in zip(axes, rasters.items()):
    vals = da.values.ravel()
    counts = {c: int((vals == c).sum()) for c in ORDER}

    bars = ax.bar(
        [MOD_CODE[c] for c in ORDER],
        [counts[c] for c in ORDER],
        color=[COLORS[c] for c in ORDER],
        edgecolor="black",
        linewidth=0.4,
    )
    for bar, c in zip(bars, ORDER):
        if counts[c] > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(counts.values()) * 0.02,
                f"{counts[c]:,}",
                ha="center",
                va="bottom",
                fontsize=7,
            )

    ax.set_title(f"{comp}  ({COMPOSITES[comp]})\npeak {peak_dates[comp]}", fontsize=9, fontweight="bold")
    ax.set_ylabel("Pixel count")
    ax.tick_params(axis="x", rotation=30, labelsize=7)

fig.suptitle("MODIS pixel-code distribution by composite — Valencia 2024", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Side‑by‑side spatial comparison

Each panel shows the same AOI as seen through a different composite.
The legend uses the canonical MODIS pixel codes.

In [6]:
from matplotlib.colors import ListedColormap, BoundaryNorm

CMAP = ListedColormap([COLORS[c] for c in ORDER])
NORM = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 255.5], CMAP.N)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (comp, da) in zip(axes, rasters.items()):
    im = da.plot.imshow(
        ax=ax, cmap=CMAP, norm=NORM, add_colorbar=False,
        interpolation="nearest",
    )
    ax.set_title(f"{comp}  ({COMPOSITES[comp]})", fontsize=9, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("")

# Shared legend
patches = [mpatches.Patch(color=COLORS[c], label=f"{c}: {MOD_CODE[c]}") for c in ORDER]
fig.legend(handles=patches, loc="lower center", ncol=5, fontsize=7.5, frameon=True, edgecolor="gray")

fig.suptitle("MODIS flood classification — Valencia 2024 peak date", fontsize=11, fontweight="bold")
plt.tight_layout(rect=[0, 0.08, 1, 0.95])
plt.show()

## 5. Difference maps

Where do the composites disagree? We map:
- **F2 − F1C**: pixels where the 2‑day composite changes the classification
  relative to the cloud‑screened 1‑day composite.
- **F3 − F2**: pixels where the 3‑day composite differs from the 2‑day.

These highlight where cloud‑shadow suppression is actively changing the result.

In [7]:
DIFF_LABELS = {
    -1: "F1C flood, F2 not",
    0: "Same code",
    1: "F2 flood, F1C not",
    -2: "Other change",
}

def diff_map(da_a: xr.DataArray, da_b: xr.DataArray, label_a: str, label_b: str):
    """Return a coded difference array: -1 = flood in A but not B; +1 = flood in B but not A."""
    a = da_a.values.astype(np.int16)
    b = da_b.values.astype(np.int16)
    diff = np.zeros_like(a, dtype=np.int16)

    # Flood code is 3 (unusual flood).  We also consider 2 (recurring).
    flood_a = (a == 2) | (a == 3)
    flood_b = (b == 2) | (b == 3)

    diff[flood_a & ~flood_b] = -1                # flood in A only
    diff[~flood_a & flood_b] = 1                 # flood in B only
    diff[(a != b) & (diff == 0)] = -2            # other code change (e.g. 255 → 0)
    return diff

diff_f2_f1c = diff_map(rasters["F2"], rasters["F1C"], "F2", "F1C")
diff_f3_f2  = diff_map(rasters["F3"], rasters["F2"],  "F3", "F2")

DIFF_CMAP = ListedColormap(["#e74c3c", "#95a5a6", "#f5f5f5", "#27ae60"])
DIFF_NORM = BoundaryNorm([-2.5, -1.5, -0.5, 0.5, 1.5], DIFF_CMAP.N)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

for ax, arr, title in [
    (ax1, diff_f2_f1c, "F2 − F1C\n(2‑day vs 1‑day cloud‑screened)"),
    (ax2, diff_f3_f2,  "F3 − F2\n(3‑day vs 2‑day)"),
]:
    im = ax.imshow(arr, cmap=DIFF_CMAP, norm=DIFF_NORM, interpolation="nearest", origin="upper")
    ax.set_title(title, fontsize=9, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("")

# Count stats
for name, arr in [("F2-F1C", diff_f2_f1c), ("F3-F2", diff_f3_f2)]:
    n_flood_added = int((arr == 1).sum())
    n_flood_lost   = int((arr == -1).sum())
    n_other        = int((arr == -2).sum())
    print(f"{name}:  +flood={n_flood_added:,}  −flood={n_flood_lost:,}  other={n_other:,}")

diff_patches = [
    mpatches.Patch(color="#27ae60", label="Flood added by longer composite"),
    mpatches.Patch(color="#e74c3c", label="Flood removed by longer composite"),
    mpatches.Patch(color="#95a5a6", label="Other code change (e.g. 255 → 0)"),
    mpatches.Patch(color="#f5f5f5", label="Same classification"),
]
fig.legend(handles=diff_patches, loc="lower center", ncol=4, fontsize=7.5, frameon=True, edgecolor="gray")

fig.suptitle("Where composites disagree — Valencia 2024", fontsize=11, fontweight="bold")
plt.tight_layout(rect=[0, 0.10, 1, 0.95])
plt.show()

## 6. Summary statistics

One table that captures the trade‑off: flood extent vs data coverage.

In [8]:
rows = []
for comp, da in rasters.items():
    vals = da.values
    total = vals.size
    rows.append({
        "Composite": comp,
        "Description": COMPOSITES[comp],
        "Peak date": peak_dates[comp],
        "Flood (code 3)": f"{100 * (vals == 3).sum() / total:.2f} %",
        "Insufficient (code 255)": f"{100 * (vals == 255).sum() / total:.2f} %",
        "Recurring flood (2)": f"{100 * (vals == 2).sum() / total:.2f} %",
        "Surface water (1)": f"{100 * (vals == 1).sum() / total:.2f} %",
    })

summary = pd.DataFrame(rows)
display(summary.style.set_caption("MODIS composite comparison — Valencia 2024").set_table_styles([
    {"selector": "caption", "props": [("font-weight", "bold"), ("font-size", "11px")]},
    {"selector": "th", "props": [("font-size", "9px")]},
    {"selector": "td", "props": [("font-size", "9px")]},
]))

## 7. Interpretation

What the plots tell us (fill in after execution):

- **F1C vs F2**: F1C has the sharpest temporal footprint but retains some
  cloud-shadow false-positives because 1‑day compositing does not suppress
  them. F2 requires ≥ 2 water detections over 2 days, so transient cloud
  shadows (which drift) are filtered out. Expect F1C to show more "flood"
  pixels — some of which are cloud-shadow artefacts.
- **F2 vs F3**: F3 applies the same logic over 3 days, further suppressing
  cloud shadows but also potentially missing short‑duration flood signals.
  The difference is usually small for clear‑sky events; larger when clouds
  persist over multiple days.
- **Insufficient‑data (255)**: Clouds that are masked as 255 in F1C may have
  water detections poking through in F2/F3 (because the compositing rule
  allows water to overwrite 255 — see [overview.md § Insufficient-data
  flag](../../docs/modis/overview.md#insufficient-data-flag-255)).

### When to use which

| Composite | Best for |
|-----------|----------|
| **F1C** | Rapid response, single‑day event characterisation (accept some noise) |
| **F2** | Event‑scale flood mapping (recommended default — balanced) |
| **F3** | Cloudy regions, multi‑day events, conservative flood masks |